# baseline_colab.ipynb

**Repo**: `scikit-image/scikit-image`
**Baseline tag**: `v0.18.3` (released 2021-03-31; ≥ 12 months old)
**Hot path**: `skimage.measure.regionprops_table`

## What this notebook does
1. Print Colab runtime info (CPU, RAM, Python version).
2. Clone scikit-image at `v0.18.3` and install it.
3. Build the deterministic workload (random non-overlapping disks +
   connected-component labelling + intensity image).
4. Run the unmodified `regionprops_table` once and save the resulting
   dict as the **golden output** to `outputs/golden_output.npz`.
5. Run the relevant existing test suite
   (`skimage/measure/tests/test_regionprops.py`) and save the set of
   passing tests to `outputs/baseline_tests.json` — that's the
   correctness reference `tests.ipynb` will compare against.
6. Measure baseline timing of the workload with the agreed
   methodology: 2 warmups + 7 measured runs, report median + IQR.
   Save to `outputs/benchmark_results.json`.

Why `regionprops_table` is meaningful: it's the canonical entry
point for extracting per-region scalar measurements from a labelled
image — the hot path in cell-counting, particle analysis, and most
bio-image pipelines. The version in `v0.18.3` computes every
property region-by-region in a Python loop; for thousands of small
regions that loop overhead dominates wall time.


## 1. Runtime info

In [ ]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version


## 1.5 Config — clone this submission repo

In [ ]:
# ====================================================================
# CONFIG (only thing the evaluator may need to change)
# --------------------------------------------------------------------
# Set the URL of THIS submission's GitHub repo (the one containing
# baseline_colab.ipynb, candidate_colab.ipynb, tests.ipynb, etc.).
# If you set the env var SUBMISSION_REPO_URL before opening Colab,
# this falls through to that value.
# ====================================================================
import os
SUBMISSION_REPO_URL = os.environ.get(
    "SUBMISSION_REPO_URL",
    "https://github.com/OWNER/REPO.git"   # <-- EDIT THIS if not set via env
)
SUBMISSION_DIR = "/content/submission"

if "OWNER/REPO" in SUBMISSION_REPO_URL:
    print("=" * 70)
    print("WARNING: SUBMISSION_REPO_URL is the placeholder.")
    print("Edit this cell to point at your fork, OR set:")
    print("  os.environ['SUBMISSION_REPO_URL'] = "
          "'https://github.com/yourname/yourrepo.git'")
    print("before re-running this cell.")
    print("=" * 70)
    raise SystemExit(0)

if not os.path.isdir(SUBMISSION_DIR):
    !git clone --depth 1 $SUBMISSION_REPO_URL $SUBMISSION_DIR

import sys
for p in (SUBMISSION_DIR,
          os.path.join(SUBMISSION_DIR, "workloads"),
          os.path.join(SUBMISSION_DIR, "benchmark"),
          os.path.join(SUBMISSION_DIR, "patches")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("Submission repo at:", SUBMISSION_DIR)


## 2. Repository setup

In [ ]:
import os
WORK = "/content/work"
BASELINE_DIR = os.path.join(WORK, "scikit-image-baseline")
os.makedirs(WORK, exist_ok=True)

if not os.path.isdir(BASELINE_DIR):
    !git clone --depth 1 --branch v0.18.3 https://github.com/scikit-image/scikit-image.git {BASELINE_DIR}

# Record the actual SHA we built (for reward.json).
BASELINE_SHA = !git -C {BASELINE_DIR} rev-parse HEAD
BASELINE_SHA = BASELINE_SHA[0].strip()
print("baseline SHA:", BASELINE_SHA)
with open("/content/baseline_sha.txt", "w") as f:
    f.write(BASELINE_SHA)


In [ ]:
# ---- Environment pinning (avoids dependency drift in Colab) -------------
# scikit-image v0.18.3 needs numpy < 1.24 and a compatible scipy.
!pip install -q --upgrade pip setuptools wheel
!pip install -q "numpy==1.23.5" "scipy==1.10.1" "cython<3" pytest line_profiler
!pip install -q networkx imageio tifffile pillow PyWavelets

import importlib, sys
for mod in ("numpy", "scipy"):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
import numpy as np, scipy
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)


In [ ]:
# Build & install scikit-image v0.18.3 (editable install so the patch
# in candidate_colab.ipynb can take effect via apply_optimization.py).
%cd $BASELINE_DIR
!pip install -q -e . 2>&1 | tail -10
%cd /content
import importlib, sys
for k in [k for k in sys.modules if k.startswith('skimage')]:
    del sys.modules[k]
import skimage
print("skimage version:", skimage.__version__)
assert skimage.__version__ == "0.18.3", "Expected scikit-image 0.18.3"


## 3. Build the workload

Non-overlapping random disks on a 4096x4096 canvas — fixed seed,
deterministic, designed so connected-component labelling produces
a few thousand distinct regions. This is realistic for cell /
particle imaging.


In [ ]:
from generate_workload import build_workload, WORKLOAD_PROPERTIES
import time, numpy as np
t0 = time.perf_counter()
labels, intensity, n_regions = build_workload()
print(f"workload built in {time.perf_counter()-t0:.2f}s | "
      f"shape={labels.shape}, n_regions={n_regions}")
os.makedirs("/content/outputs", exist_ok=True)
np.savez_compressed("/content/outputs/workload.npz",
                    labels=labels, intensity=intensity)


## 4. Golden output — run regionprops_table once and save

In [ ]:
from skimage.measure import regionprops_table
out = regionprops_table(labels, intensity_image=intensity,
                        properties=list(WORKLOAD_PROPERTIES))
# Save as .npz so the candidate notebook can compare column-wise.
np.savez_compressed("/content/outputs/golden_output.npz", **out)
print("golden_output.npz columns:", sorted(out.keys()))
print("first 3 rows of 'area':", out["area"][:3])


## 5. Existing test suite — record the pass-set

We run only the test module that exercises `regionprops` /
`regionprops_table`. Running the whole scikit-image suite would
blow our 75-minute budget and isn't relevant — the patch only
touches `regionprops_table`.


In [ ]:
import subprocess, json
# --tb=no keeps the report parseable; we only need pass/fail per test.
proc = subprocess.run(
    ["pytest", os.path.join(BASELINE_DIR,
     "skimage/measure/tests/test_regionprops.py"),
     "-v", "--tb=no", "-q", "--no-header"],
    capture_output=True, text=True,
)
print(proc.stdout[-2000:])
print("STDERR:", proc.stderr[-500:])
passed = []
failed = []
for line in proc.stdout.splitlines():
    line = line.strip()
    if " PASSED" in line:
        passed.append(line.split(" PASSED")[0].strip())
    elif " FAILED" in line:
        failed.append(line.split(" FAILED")[0].strip())
baseline_tests = {"passed": sorted(passed), "failed": sorted(failed)}
with open("/content/outputs/baseline_tests.json", "w") as f:
    json.dump(baseline_tests, f, indent=2)
print(f"\n{len(passed)} tests passed, {len(failed)} failed at baseline.")


## 6. Benchmark baseline

Methodology: high-precision `time.perf_counter()`, 2 warmups, 7
measured runs, report median + IQR.


In [ ]:
from benchmark_utils import bench, write_json, collect_environment

def run_baseline():
    return regionprops_table(labels, intensity_image=intensity,
                             properties=list(WORKLOAD_PROPERTIES))

baseline_stats = bench(run_baseline, label="baseline",
                       n_warmup=2, n_measured=7)
env = collect_environment()
write_json("/content/outputs/benchmark_results.json", {
    "baseline_time_s": baseline_stats,
    "environment": env,
    "baseline_sha": BASELINE_SHA,
})
print("\nSaved /content/outputs/benchmark_results.json")
print(f"Baseline median: {baseline_stats['median']:.3f}s "
      f"(IQR {baseline_stats['iqr']:.3f}s)")
